# Подготовка сырых данных

Рассмотрим задачу обнаружения ответов LLM, содержащих галлюцинации

По промпту, ответу и вероятностям выходных токенов будем определять, является ли ответ галлюцинацией или нет (бинарная классификация)

В данном ноутбуке преобразуем сырые данные по генерациям модели [Mistral](https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.1-GGUF) на задачах из [FLAN Collection](https://www.kaggle.com/datasets/thedevastator/open-orca-augmented-flan-dataset) представленные в датасете на [Kaggle](https://www.kaggle.com/datasets/lucamassaron/genai-hallucinations/)

In [1]:
%load_ext autoreload
%autoreload 2

## Импорт необходимых библиотек

In [2]:
import pandas as pd

In [3]:
from core.data_preparation.utils import *
from core.hallucinations_judging.utils import *

Для воспроизводимости экспериментов

In [4]:
init_random_seed(316)

In [5]:
data_path = 'data'
queries_path = 'data/mistral-queries'

## Загрузка и обработка данных

Загрузим набор json-файлов, предоставленный в [датасете](https://www.kaggle.com/datasets/lucamassaron/genai-hallucinations/) и превратим их в единый датафрейм

In [6]:
queries_df = build_queries_df(queries_path)
queries_df.head(10)

,id,Prompt,expected_answer,model,result_id,temperature,top_p,completion_tokens,tokens,Answer,token_logprobs,finish_reason
0,t0.666299,[INST] You are an AI assistant. User will you ...,- indirectly impacting a step of the process\n...,mistral-7b-instruct-v0.1.Q5_K_M.gguf,0,0.0,0.95,69,"[ , -, The, number, of, birds, that, ma...",- The number of birds that mature and fly aw...,"[-1.081483447869355, -0.22320987096984157, -1....",stop
1,t0.666299,[INST] You are an AI assistant. User will you ...,- indirectly impacting a step of the process\n...,mistral-7b-instruct-v0.1.Q5_K_M.gguf,1,0.8,0.95,100,"[ -, Ind, irect, ly, impact, ing, a, step,...",- Indirectly impacting a step of the process\...,"[-1.194560475150154, -2.6009601144127554, -0.0...",stop
2,t0.666299,[INST] You are an AI assistant. User will you ...,- indirectly impacting a step of the process\n...,mistral-7b-instruct-v0.1.Q5_K_M.gguf,2,0.8,0.95,9,"[ -, directly, impact, ing, a, step, of, ...",- directly impacting a step of the process,"[-1.194560475150154, -1.8035605935387318, -0.0...",stop
3,t0.666299,[INST] You are an AI assistant. User will you ...,- indirectly impacting a step of the process\n...,mistral-7b-instruct-v0.1.Q5_K_M.gguf,3,0.8,0.95,10,"[ -, Direct, ly, impact, ing, a, step, of...",- Directly impacting a step of the process,"[-1.194560475150154, -2.1106236962609, -0.0037...",stop
4,t0.666299,[INST] You are an AI assistant. User will you ...,- indirectly impacting a step of the process\n...,mistral-7b-instruct-v0.1.Q5_K_M.gguf,4,0.8,0.95,34,"[ The, answer, depends, on, the, specific...",The answer depends on the specific scenario p...,"[-2.8148645828161696, -2.301702688153816, -2.3...",stop
5,t0.666299,[INST] You are an AI assistant. User will you ...,- indirectly impacting a step of the process\n...,mistral-7b-instruct-v0.1.Q5_K_M.gguf,5,0.8,0.95,170,"[ The, direct, impact, on, the, process, ...",The direct impact on the process would be if ...,"[-2.8148645828161696, -3.1673585872016674, -0....",stop
6,t0.666299,[INST] You are an AI assistant. User will you ...,- indirectly impacting a step of the process\n...,mistral-7b-instruct-v0.1.Q5_K_M.gguf,6,0.8,0.95,101,"[ , -, The, process, of, butter, fly, me...",- The process of butterfly metamorphosis is ...,"[-1.104686207572029, -0.22462651985117135, -1....",stop
7,t0.666299,[INST] You are an AI assistant. User will you ...,- indirectly impacting a step of the process\n...,mistral-7b-instruct-v0.1.Q5_K_M.gguf,7,0.8,0.95,10,"[ Ind, irect, ly, impact, ing, a, step, of...",Indirectly impacting a step of the process,"[-2.8192333648596266, -0.0004916830827148915, ...",stop
8,t0.666299,[INST] You are an AI assistant. User will you ...,- indirectly impacting a step of the process\n...,mistral-7b-instruct-v0.1.Q5_K_M.gguf,8,0.8,0.95,36,"[ -, The, adult, butter, fly, emer, ges, ...",- The adult butterfly emerges out of the chry...,"[-1.194560475150154, -1.668515541964025, -3.37...",stop
9,t0.666299,[INST] You are an AI assistant. User will you ...,- indirectly impacting a step of the process\n...,mistral-7b-instruct-v0.1.Q5_K_M.gguf,9,0.8,0.95,102,"[ -, Direct, ly, impact, ing, a, step, of...",- Directly impacting a step of the process - ...,"[-1.194560475150154, -2.1106236962609, -0.0037...",stop


Видно, что в получившемся фрейме есть записи с дублирующимися `id` но различающимися значениями. Посмотрим, сколько записей приходится на один `id`, идентифицирующий пару **Prompt-Expected Answer**

In [7]:
id_counts = queries_df.id.value_counts()
id_counts.describe().to_frame().T

,count,mean,std,min,25%,50%,75%,max
count,18671.0,21.0,0.0,21.0,21.0,21.0,21.0,21.0


Из описательных статистик можно сделать предположить что на каждый промпт было сгенерировано по 21-му ответу. Проверим это

In [8]:
id_counts.unique()

array([21])

Действительно, для каждого промпта было сгенерировано по 21-му ответу, что избыточно для решения задачи галлюцинаций, а также ведет к излишней затратности процесса разметки. Сократим данное число

In [9]:
queries_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 392091 entries, 0 to 392090
Data columns (total 12 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   id                 392091 non-null  object 
 1   Prompt             392091 non-null  object 
 2   expected_answer    392091 non-null  object 
 3   model              392091 non-null  object 
 4   result_id          392091 non-null  int64  
 5   temperature        392091 non-null  float64
 6   top_p              392091 non-null  float64
 7   completion_tokens  392091 non-null  int64  
 8   tokens             392091 non-null  object 
 9   Answer             392091 non-null  object 
 10  token_logprobs     392091 non-null  object 
 11  finish_reason      392091 non-null  object 
dtypes: float64(2), int64(2), object(8)
memory usage: 35.9+ MB


В датасете более `390 000` записей, что также излишне для обучения легковесной модели

## Сокращение размера датасета

Параметры `temperature` и `top_p` во всех генерациях практически идентичны. Посмотрим как они менялись при построении датасета

In [10]:
queries_df.temperature.value_counts().to_frame()

,count
temperature,
0.8,373420
0.0,18671


In [11]:
queries_df.top_p.value_counts().to_frame()

,count
top_p,
0.95,392091


Видно, что значение `top_p` во всех генерациях одинаково, а `temperature` принимает значения `0` и `0.8`, что соответствует максимально детерминированному ответу и ответам с большей степенью "свободы" модели

Сохраним все ответы с нулевой температурой и отфильтруем получившиеся при идентичных параметрах генерации 

Для фильтрации ответов введем критерий "риска", который будет основан на вероятностях сгенерированных токенов ответа (столбец `token_logprobs`)

Пусть $\{\ell_i\}_{i=1}^{n}$ — логарифмы вероятностей (logprob) токенов ответа,
где $n$ — количество токенов.

Определим:

$$
\mu = \frac{1}{n}\sum_{i=1}^{n} \ell_i
$$

— среднее значение logprob,

$$
\sigma = \sqrt{\frac{1}{n}\sum_{i=1}^{n} (\ell_i - \mu)^2}
$$

— стандартное отклонение logprob,

$$
f = \frac{1}{n}\sum_{i=1}^{n} \mathbf{1}(\ell_i < \tau)
$$

— долю токенов с низкой уверенностью,
где $\tau$ — порог уверенности (confidence threshold),
а $\mathbf{1}(\cdot)$ — индикаторная функция.

Тогда функция риска определяется как:

$$
\mathrm{Risk}(\{\ell_i\}) =
\frac{
-\alpha\mu + \beta\,\sigma + \gamma\,f
}{
\log(n + 1)
}
$$

где $\alpha, \beta, \gamma \ge 0$ — настраиваемые коэффициенты,
определяющие вклад соответствующих компонент в итоговый риск:

- $\alpha$ регулирует чувствительность к средней уверенности модели (идет со знаком минус, чтобы максимизировать значение риска при низком среднем значении logprobs) — в текущей реализации равен $1$;
- $\beta$ задаёт важность нестабильности предсказаний — в текущей реализации равен $0.5$;
- $\gamma$ контролирует вклад доли низкоуверенных токенов — в текущей реализации равен $0.3$.

Таким образом:

- $-\mu$ увеличивает риск при низкой средней уверенности модели;
- $\sigma$ повышает риск при высокой нестабильности предсказаний;
- $f$ усиливает вклад токенов с особенно низкой вероятностью;
- деление на $\log(n+1)$ нормализует риск по длине ответа.



Вычислим значение риска для каждой генерации и для каждой пары **Prompt-Expected Answer** оставим следующие генерации:

- генерацию, полученную при нулевой температуре;
- генерацию с минимальным риском (предположительно не являющуюся галлюцинацией);
- генерацию с максимальным риском (предположительно галлюцинация);
- генерацию с наиболее близким риском к медиане (наиболее интересные для классификации примеры)

In [12]:
filtered_df = filter_by_risk_and_temperature(queries_df)
filtered_df.head(10)

,id,Prompt,expected_answer,model,result_id,temperature,top_p,completion_tokens,tokens,Answer,token_logprobs,finish_reason,_risk
0,t0.666299,[INST] You are an AI assistant. User will you ...,- indirectly impacting a step of the process\n...,mistral-7b-instruct-v0.1.Q5_K_M.gguf,0,0.0,0.95,69,"[ , -, The, number, of, birds, that, ma...",- The number of birds that mature and fly aw...,"[-1.081483447869355, -0.22320987096984157, -1....",stop,0.183465
1,t0.666299,[INST] You are an AI assistant. User will you ...,- indirectly impacting a step of the process\n...,mistral-7b-instruct-v0.1.Q5_K_M.gguf,8,0.8,0.95,36,"[ -, The, adult, butter, fly, emer, ges, ...",- The adult butterfly emerges out of the chry...,"[-1.194560475150154, -1.668515541964025, -3.37...",stop,0.157626
2,t0.666299,[INST] You are an AI assistant. User will you ...,- indirectly impacting a step of the process\n...,mistral-7b-instruct-v0.1.Q5_K_M.gguf,12,0.8,0.95,11,"[ It, depends, on, the, specific, s, of, ...",It depends on the specifics of the scenario p...,"[-4.402656025687263, -2.012412148955237, -0.01...",stop,0.765049
3,t0.666299,[INST] You are an AI assistant. User will you ...,- indirectly impacting a step of the process\n...,mistral-7b-instruct-v0.1.Q5_K_M.gguf,3,0.8,0.95,10,"[ -, Direct, ly, impact, ing, a, step, of...",- Directly impacting a step of the process,"[-1.194560475150154, -2.1106236962609, -0.0037...",stop,0.287458
4,cot.37485,[INST] You are an AI assistant that helps peop...,"Yes, the hypothesis is entailed by the premise...",mistral-7b-instruct-v0.1.Q5_K_M.gguf,0,0.0,0.95,938,"[\n, \n, \n, \n, \n, \n, \n, \n, \n, \n, \n, \...",\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n...,"[-1.3593680472128356, -1.6231154943046324, -1....",length,0.008531
5,cot.37485,[INST] You are an AI assistant that helps peop...,"Yes, the hypothesis is entailed by the premise...",mistral-7b-instruct-v0.1.Q5_K_M.gguf,13,0.8,0.95,51,"[ , Den, ny, asked, :, Prem, ise, :,...","Denny asked: Premise: ""Two men are pu...","[-4.115206902905787, -1.3768656648957738, -0.0...",stop,0.137743
6,cot.37485,[INST] You are an AI assistant that helps peop...,"Yes, the hypothesis is entailed by the premise...",mistral-7b-instruct-v0.1.Q5_K_M.gguf,8,0.8,0.95,25,"[ , Den, ny, asked, :, Yes, or, no,...","Denny asked: Yes or no.\nLe: No, we ne...","[-2.957195466443385, -0.6023525322208485, -0.0...",stop,0.563410
7,cot.37485,[INST] You are an AI assistant that helps peop...,"Yes, the hypothesis is entailed by the premise...",mistral-7b-instruct-v0.1.Q5_K_M.gguf,2,0.8,0.95,66,"[\n, The, hypothesis, is, not, necessarily...",\nThe hypothesis is not necessarily entailed b...,"[-1.318741982861842, -2.7469867811466955, -1.2...",stop,0.261395
8,flan.274087,[INST] You are an AI assistant. User will you ...,"In Yemen, 56-year-old Australian Craig Bruce M...",mistral-7b-instruct-v0.1.Q5_K_M.gguf,0,0.0,0.95,126,"[\n, A, , 5, 6, -, year, -, old, Australian,...",\nA 56-year-old Australian man was kidnapped i...,"[-0.2469331012480379, -1.5850518182718765, -0....",stop,0.082767
9,flan.274087,[INST] You are an AI assistant. User will you ...,"In Yemen, 56-year-old Australian Craig Bruce M...",mistral-7b-instruct-v0.1.Q5_K_M.gguf,1,0.8,0.95,217,"[\n, A, , 5, 6, -, year, -, old, Australian,...","\nA 56-year-old Australian man, Craig Bruce Mc...","[-0.2525840373909285, -1.59851492802654, -0.59...",stop,0.095849


In [13]:
filtered_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69451 entries, 0 to 69450
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 69451 non-null  object 
 1   Prompt             69451 non-null  object 
 2   expected_answer    69451 non-null  object 
 3   model              69451 non-null  object 
 4   result_id          69451 non-null  int64  
 5   temperature        69451 non-null  float64
 6   top_p              69451 non-null  float64
 7   completion_tokens  69451 non-null  int64  
 8   tokens             69451 non-null  object 
 9   Answer             69451 non-null  object 
 10  token_logprobs     69451 non-null  object 
 11  finish_reason      69451 non-null  object 
 12  _risk              69451 non-null  float64
dtypes: float64(3), int64(2), object(8)
memory usage: 6.9+ MB


С помощью примененной эвристики размер датасета был сокращен примерно в **6** раз

Также удалим все ответы в которых нет текста, интересующего в рамках решаемой задачи (в ответе нет ни одной цифры либо латинского / кириллического символа)

In [14]:
filtered_df = remove_non_text_answers(filtered_df)

Удалено неинформативных записей: 1056


Также удалим неинформативные для дальнейшей работы столбцы

In [15]:
filtered_df.drop(['model', 'result_id', 'tokens', 'id'], axis=1, inplace=True)

И сохраним получившийся датасет, с которым будем работать на следующих шагах

In [16]:
filtered_df.to_csv(f"{data_path}/filtered_queries.csv")